# Subthread host llama.cpp server
## make llama-server executable (current version 8182)

In [1]:
!cp /kaggle/input/datasets/tnthnhneee/llama-server-gpu-8182/llama-server /kaggle/working/llama-server
!chmod +x /kaggle/working/llama-server

In [2]:
!/kaggle/working/llama-server --version

ggml_cuda_init: found 2 CUDA devices:
  Device 0: Tesla T4, compute capability 7.5, VMM: yes
  Device 1: Tesla T4, compute capability 7.5, VMM: yes
version: 8182 (05728db18)
built with GNU 11.4.0 for Linux x86_64


## HOST LLAMA-SERVER

In [3]:
import os
import shutil
import subprocess
import threading

env = os.environ.copy()

LLAMA_ARG_PORT="8000"

# Build the command
command = [
    "./llama-server",
    "-m", "/kaggle/input/datasets/leqvinh/huihui-qwen3-5-27b-claude-4-6-opus-abliterated/Huihui-Qwen3.5-27B-Claude-4.6-Opus-abliterated.i1-Q4_K_S.gguf",

    # === BẮT BUỘC CHO VISION (VL) ===
    # 🟢 SỬA LỖI 1: Cập nhật đúng tên file mmproj
    # "--mmproj", "/kaggle/input/datasets/leqvinh/qwen3-vl-4b-instruct-bf16-gguf/mmproj-BF16.gguf",
    
    "--port", "8000",
    
    # === SAMPLING PARAMETERS (Quan trọng cho chất lượng code) ===
    "--seed", "3407",                    # Seed cố định cho reproducibility
    "--temp", "0.6",                     # Giảm xuống 0.6 cho code chính xác hơn (Unsloth khuyến nghị)
    "--top-p", "0.95",                   # Nucleus sampling - giữ lại 95% probability mass
    "--top-k", "20",                     # Giới hạn 20 token ứng viên - giảm độ ngẫu nhiên
    "--min-p", "0.0",                    # Cho phép token có xác suất thấp nhất
    "--repeat-penalty", "1.0",           # Tắt repeat penalty cho code (theo Unsloth)
    
    # === PERFORMANCE OPTIMIZATION ===
    "--flash-attn", "on",                # Bật Flash Attention - tăng tốc đáng kể
    "--kv-unified",                      # Unified KV cache - tiết kiệm VRAM
    "--cache-type-k", "q8_0",            # KV cache quantization (có thể thử bf16 nếu đủ VRAM)
    "--cache-type-v", "q8_0",
    
    # === BATCHING (Tối ưu cho throughput) ===
    # 🟢 SỬA LỖI 2: Giảm batch size để tránh OOM khi nạp ảnh
    "--batch-size", "256000",              # Logical batch size 
    "--ubatch-size", "2048",              # Physical batch size
    
    # === GPU OFFLOADING ===
    "--n_gpu_layers", "99",              # Offload tối đa lên GPU
    "--tensor-split", "1,1",             # Chia đều 2 GPU
    
    # === CONTEXT ===
    # 🟢 SỬA LỖI 3: Giảm Context xuống 32K (128K sẽ gây lỗi cudaMalloc failed trên T4)
    "--ctx-size", "256000",               
    
    # === SERVER FEATURES ===
    "--parallel", "1",                   # 1 slot cho single-user vibe coding
    "--cont-batching",                   # Continuous batching (mặc định bật)
    "--jinja",                           # BẮT BUỘC cho tool calling/function calling
    
    # === OPTIONAL: Disable thinking mode cho code nhanh hơn ===
    "--chat-template-kwargs", '{"enable_thinking":false}',
    
    # === LOGGING ===
    "--verbose-prompt"
]

print("Đang khởi động llama-server...")

llama_process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,              # decode automatically
    encoding="utf-8",       # or another encoding
    errors="replace",       # avoid crashes on bad bytes
    env=env                 # Truyền env vào process
)

Đang khởi động llama-server...


## Expose local port to the internet using **ngrok**

Add-ons > Secrets -> Add `NGROK_TOKEN`

In [ ]:
!pip install pyngrok --quiet
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    ngrok_token = user_secrets.get_secret("NGROK_TOKEN")
    !ngrok config add-authtoken {ngrok_token}
except Exception as e:
    print("NGROK_TOKEN not found in environment variables. Add-ons > Secrets -> Add NGROK_TOKEN")

from pyngrok import ngrok

# Open ngrok tunnel to port $PORT
public_url = ngrok.connect(LLAMA_ARG_PORT)
print(f"Public API endpoint: {public_url}")

# Stream logs in real time
for line in llama_process.stdout:
    print(line, end="")

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml                                
Public API endpoint: NgrokTunnel: "https://snowcapped-denice-unprudent.ngrok-free.dev" -> "http://localhost:8000"
ggml_cuda_init: found 2 CUDA devices:
  Device 0: Tesla T4, compute capability 7.5, VMM: yes
  Device 1: Tesla T4, compute capability 7.5, VMM: yes
build: 8182 (05728db18) with GNU 11.4.0 for Linux x86_64
system info: n_threads = 2, n_threads_batch = 2, total_threads = 4

system_info: n_threads = 2 (n_threads_batch = 2) / 4 | CUDA : ARCHS = 750 | USE_GRAPHS = 1 | PEER_MAX_BATCH_SIZE = 128 | CPU : SSE3 = 1 | SSSE3 = 1 | AVX = 1 | AVX2 = 1 | F16C = 1 | FMA = 1 | BMI2 = 1 | LLAMAFILE = 1 | OPENMP = 1 | REPACK = 1 | 

Running without SSL
init: using 3 threads for HTTP server
start: binding port with default address family
main: loading model
srv    load_model: loading model '/kaggle/input/datasets/leqvinh/huihui-qwen3-5-27b-claude-4-6-opus-abliterated/Huihui-Qwen3.5-27B-Claude-4